In [1]:
import pandas as pd
import pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from geoai.utils_ds.preprocessing_ops import PreProcessingOperations
from geoai.utils_ml.model_ops import ModelOperations
from geoai.utils_geo.raster_ops import RasterOperations

preprocess_ops = PreProcessingOperations()
model_ops = ModelOperations()
raster_ops = RasterOperations()


# We will create a pipeline using the following steps:

1. Load the data containing only the bands.

2. Compute indices

3. Bin and Categorize NDVI

4. Apply Log transformation to numerical features

5. Do a Polynomial transformation

6. One hot encode NDVI_binary

7. Ordinal encode NDVI_category

8. Scale to 0-1

9. Apply LDA

10. Train a logisitic regression


In [2]:
# Step 1
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR
0,350.0,542.0000,323.0,3277.0000,1975.3334
1,390.0,555.0000,380.0,3016.6667,1991.0000
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000
3,363.2,546.5000,395.0,3244.5000,2052.0000
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000


In [3]:
# Step 2
X_train = raster_ops.indices_binary_category(X_train) 
X_test = raster_ops.indices_binary_category(X_test)
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545,high_veg,veg
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227,high_veg,veg
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127,low_veg,non_veg
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438,high_veg,veg
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098,low_veg,non_veg


In [6]:
# # Define the columns that will be used in the pipeline
# numerical_columns = X_train.select_dtypes(include=["float64"]).columns.tolist()
# one_hot_encoder_columns = ["NDVI_binary"]
# ordinal_encoder_columns = ["NDVI_categorized"]
# categories = [["low_veg", "medium_veg", "high_veg"]]

In [4]:
# Step 4 and 5
log_trasformer = FunctionTransformer(func=np.log1p)  # Instantiate the log transformer
poly_transformer = PolynomialFeatures(
    degree=2
)  # Instantiate the polynomial transformer

# Create a pipeline that includes the log transformation and a polynomial transformation
pipeline_1 = Pipeline(
    steps=[("log_trasformer", log_trasformer), ("poly_transformer", poly_transformer)]
)

In [8]:
numerical_columns = X_train.select_dtypes(include=["float64"]).columns.tolist()
one_hot_encoder_columns = ["NDVI_binary"]
ordinal_encoder_columns = ["NDVI_categorized"]
categories = [["low_veg", "medium_veg", "high_veg"]]

# Step 4, 5, 6 and 7
one_hot_transformer = OneHotEncoder(dtype=int, sparse_output=False) # Instantiate the one hot transformer
ordinal_transformer = OrdinalEncoder(categories=categories, dtype=int) # Instantiate the ordinal transformer

# Create a preprocessor that includes the numerical, one hot, and ordinal transformers
pipeline_2 = ColumnTransformer(
    transformers=[
        ("numerical_transformer", pipeline_1, numerical_columns),
        ("onehot", one_hot_transformer, one_hot_encoder_columns),
        ("ordinal", ordinal_transformer, ordinal_encoder_columns),
    ],
    remainder="drop", # Drop columns not specified in the transformers
)

In [9]:
# Step 8 to 10
scaler = MinMaxScaler() # Instantiate the min max scaler
dim_reduce = LinearDiscriminantAnalysis(n_components=3) # Instantiate the linear discriminant analysis
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=1) # Instantiate the logistic regression model

# Create a pipeline that includes the preprocessor, scaler, dimensionality reduction, and model
pipeline = Pipeline(
    steps=[
        ("pipeline_2", pipeline_2),
        ("scaler", scaler),
        ("dim_reduce", dim_reduce),
        ("lr", lr),
    ]
)
pipeline

Pipeline(steps=[('pipeline_2',
                 ColumnTransformer(transformers=[('numerical_transformer',
                                                  Pipeline(steps=[('log_trasformer',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('poly_transformer',
                                                                   PolynomialFeatures())]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI']),
                                                 ('onehot',
                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['NDVI_binary']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                              'high_veg']],
                                                                 dtype=<class 'int'>),
                                                  ['NDVI_categorized'])])),
                ('scaler', MinMaxScaler()),
                ('dim_reduce', LinearDiscriminantAnalysis(n_components=3)),
                ('lr', LogisticRegression(max_iter=1000, random_state=1))])

In [11]:
# save the model using pickle
# merge the train and test datasets

X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the best hyperparameters and the whole dataset
pipeline.fit(X_all, y_all)
with open('trained_models/lda_pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


END